# Lab 9: Data Augmentation with Keras ImageDataGenerator

## Objective
Explore the effect of data augmentation on the performance of a CNN trained on the corn leaf disease dataset (4 classes). We compare training with and without augmentation using the same model architecture.

## Dataset
The dataset is structured into `train/`, `validation/`, and `test/` directories, each containing four subfolders:
- `Cercospora`
- `common_rust`
- `healthy`
- `northern_leaf_blight`

## Model Architecture (Same as Lab 8)
The CNN consists of:
- **Conv2D** layers with increasing filters (32 → 64 → 128 → 128), each followed by **MaxPooling2D**.
- **Flatten** layer.
- **Dropout** (0.5) after flattening.
- **Dense** layer with 512 units and ReLU activation.
- **Output Dense** layer with 4 units and Sigmoid activation (multiclass).

## Augmentation Parameters (Experiment 1)
- `rotation_range=40`
- `width_shift_range=0.2`
- `height_shift_range=0.2`
- `shear_range=0.2`
- `zoom_range=0.2`
- `horizontal_flip=True`

## Training
- **Loss:** `categorical_crossentropy`
- **Optimizer:** RMSprop with learning rate 1e-4
- **Metrics:** Accuracy
- **Epochs:** 10 (both experiments)
- **Callbacks:** ModelCheckpoint to save the best model based on validation loss.

## Evaluation
After both training runs, the best saved model is evaluated on the test set using a confusion matrix and classification report.

---


In [1]:
# Import necessary libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

## 1. Define Paths and Checkpoints

In [2]:
# Directory paths for training, validation, and test sets
train_dir = r'C:\\Users\\PMYLS\\Documents\\MachineLearningLaB\\Computervision\\train'
validation_dir = r'C:\\Users\\PMYLS\\Documents\\MachineLearningLaB\\Computervision\\validation'
test_dir = r'C:\\Users\\PMYLS\\Documents\\MachineLearningLaB\\Computervision\\test'

# Checkpoint file pattern (overwrites previous best model)
checkpoints = r'C:\\Users\\PMYLS\\Documents\\MachineLearningLaB\\Computervision\\E1-cp-0014-loss0.20.h5'

## 2. Build the CNN Model
The model architecture is defined using Keras Sequential API. Note the addition of a Dropout layer (0.5) after flattening.

In [3]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5),          # Regularization
    layers.Dense(512, activation='relu'),
    layers.Dense(4, activation='sigmoid')   # 4 classes
])

In [4]:
model.summary()

## 3. Compile the Model

In [5]:
model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizers.RMSprop(learning_rate=1e-4),
    metrics=['acc']
)

## 4. Experiment 1: Training with Data Augmentation
We use an augmented `ImageDataGenerator` to create variations of the training images. This helps the model generalize better.

In [6]:
# Augmented training generator
train_datagen_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Validation generator (only rescaling)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator_aug = train_datagen_aug.flow_from_directory(
    train_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical'
)

validation_generator = test_datagen.flow_from_directory(
    validation_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical'
)

### ModelCheckpoint Callback
Saves the model whenever validation loss improves (overwrites the same file).

In [7]:
checkpoint = ModelCheckpoint(
    checkpoints,
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)
callbacks = [checkpoint]

### Train with Augmentation (10 epochs)

In [8]:
# Train with augmented data
history_aug = model.fit(
    train_generator_aug,
    validation_data=validation_generator,
    steps_per_epoch=train_generator_aug.n // train_generator_aug.batch_size,
    validation_steps=validation_generator.n // validation_generator.batch_size,
    epochs=10,
    callbacks=callbacks
)

In [9]:
history_aug.history

## 5. Experiment 2: Training without Data Augmentation
For comparison, we now train the same model using only rescaling (no augmentation). Note: The model weights are reset by redefining the model or by recompiling? In the original notebook, the model was redefined earlier, but we are reusing the same model instance. However, since we are training again, we must reset the model weights. In the original code, they redefined the generators but not the model; they trained again on the same model, which would continue training from the previous weights. To properly compare, we should reinitialize the model. Since we are preserving the original logic, we will keep it as is (they trained again on the same model, which is not a fair comparison but that's what they did). We'll document this.

We'll redefine the generators without augmentation and train again for 10 epochs.

In [10]:
# Non-augmented training generator (only rescaling)
train_datagen_noaug = ImageDataGenerator(rescale=1./255)

train_generator_noaug = train_datagen_noaug.flow_from_directory(
    train_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical'
)

# Validation generator remains the same
validation_generator = test_datagen.flow_from_directory(
    validation_dir,
    target_size=(256, 256),
    batch_size=32,
    class_mode='categorical'
)

### Train without Augmentation (10 epochs)
**Note:** The model continues training from the weights learned in Experiment 1. To isolate the effect of augmentation, one should reinitialize the model. However, we follow the original notebook's flow.

In [11]:
# Train without augmentation (continuing from previous weights)
history_noaug = model.fit(
    train_generator_noaug,
    validation_data=validation_generator,
    steps_per_epoch=train_generator_noaug.n // train_generator_noaug.batch_size,
    validation_steps=validation_generator.n // validation_generator.batch_size,
    epochs=10,
    callbacks=callbacks
)

In [12]:
history_noaug.history

## 6. Visualize Training Curves
We plot accuracy and loss for the second training run (without augmentation) as shown in the original notebook. Note: This only shows the second run; to compare both, one would need to plot both histories together.

In [13]:
# Plot training and validation accuracy for the second run
acc = history_noaug.history['acc']
val_acc = history_noaug.history['val_acc']
loss = history_noaug.history['loss']
val_loss = history_noaug.history['val_loss']
epochs = range(1, len(acc) + 1)

plt.plot(epochs, acc, 'bo', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and Validation Accuracy (No Augmentation)')
plt.legend()
plt.figure()

plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and Validation Loss (No Augmentation)')
plt.legend()
plt.show()

# Save figure (optional)
# plt.savefig(r'C:\\Users\\PMYLS\\Documents\\MachineLearningLaB\\Computervision\\model_Accuracy.png')

## 7. Load Best Model and Evaluate on Test Set
Load the saved model (best from the last training run) and generate predictions on the test data. Compute confusion matrix and classification report.

In [14]:
# Load the best model saved during training (overwrites previous)
model = load_model(checkpoints)

# Test generator (only rescaling, no shuffle)
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(256, 256),
    batch_size=32,
    shuffle=False,
    class_mode='categorical'
)

# Get true labels and predictions
true_labels = test_generator.labels
pred_probs = model.predict(test_generator)
pred_labels = np.argmax(pred_probs, axis=1)

# Confusion matrix
cm = confusion_matrix(true_labels, pred_labels)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Cercospora', 'common_rust', 'healthy', 'leaf_blight']
)
disp.plot()
plt.title('Confusion Matrix on Test Set')
plt.show()

# Classification report
print(classification_report(
    true_labels,
    pred_labels,
    target_names=['Cercospora', 'common_rust', 'healthy', 'leaf_blight']
))